# 04 - 高级块操作测试

测试内容:
1. 创建各种块类型（标题、列表、代码、引用、待办）
2. 批量追加块（chunked）
3. 文档元数据读取
4. 错误处理测试

参考:
- joeseesun/qiaomu-feishu-lark-agent/feishu.py 中的 _md_to_blocks
- 飞书块类型文档: https://open.feishu.cn/document/server-docs/docs/docs/docx-v1/blocks/overview

In [ ]:
import os
import json
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(Path('../../.env'))
except ImportError:
    pass

from feishu_client import (
    FeishuClient, md_to_blocks, make_text_block, make_heading_block,
    make_code_block, extract_text_from_block, block_type_name
)

client = FeishuClient()
print("✓ 客户端初始化成功")

## 4.1 创建测试文档（各种块类型）

In [ ]:
# 创建新文档
result = client.api('POST', '/docx/v1/documents', json_data={
    'title': '块类型测试文档'
})

ADV_DOC_ID = result['document']['document_id']
print(f"文档 ID: {ADV_DOC_ID}")
print(f"URL: https://feishu.cn/docx/{ADV_DOC_ID}")

In [ ]:
# 手动构建各种块类型
blocks = [
    # 标题
    make_heading_block("一级标题", level=1),
    make_heading_block("二级标题", level=2),
    make_heading_block("三级标题", level=3),
    
    # 普通文本
    make_text_block("这是一段普通文本，用于测试文本块的基本功能。"),
    
    # 无序列表
    {
        'block_type': 12,
        'bullet': {
            'elements': [{'text_run': {'content': '无序列表项 1', 'text_element_style': {}}}]
        }
    },
    {
        'block_type': 12,
        'bullet': {
            'elements': [{'text_run': {'content': '无序列表项 2', 'text_element_style': {}}}]
        }
    },
    
    # 有序列表
    {
        'block_type': 13,
        'ordered': {
            'elements': [{'text_run': {'content': '第一步', 'text_element_style': {}}}]
        }
    },
    {
        'block_type': 13,
        'ordered': {
            'elements': [{'text_run': {'content': '第二步', 'text_element_style': {}}}]
        }
    },
    
    # 代码块
    make_code_block("def hello():\n    print('Hello Feishu!')"),
    
    # 引用
    {
        'block_type': 15,
        'quote': {
            'elements': [{'text_run': {'content': '这是一段引用文本', 'text_element_style': {}}}]
        }
    },
    
    # 待办
    {
        'block_type': 17,
        'todo': {
            'elements': [{'text_run': {'content': '待办事项：测试飞书 API', 'text_element_style': {}}}],
            'style': {},
            'checked': False
        }
    },
    
    # 分割线
    {
        'block_type': 22,
        'divider': {}
    },
    
    # 更多文本
    make_text_block("文档结束。以上展示了飞书 docx v1 API 支持的主要块类型。"),
]

print(f"准备追加 {len(blocks)} 个块")

In [ ]:
# 分批追加（每批最多 50 个，避免 API 限制）
chunk_size = 50
for i in range(0, len(blocks), chunk_size):
    chunk = blocks[i:i+chunk_size]
    client.api(
        'POST',
        f'/docx/v1/documents/{ADV_DOC_ID}/blocks/{ADV_DOC_ID}/children',
        json_data={'children': chunk, 'index': i}
    )
    print(f"✓ 已追加块 {i+1} - {min(i+chunk_size, len(blocks))}")

print(f"\n✓ 全部 {len(blocks)} 个块追加完成")

## 4.2 验证文档内容

In [ ]:
# 读取纯文本
result = client.api('GET', f'/docx/v1/documents/{ADV_DOC_ID}/raw_content')
print("=== 纯文本内容 ===")
print(result.get('content', '无内容'))

In [ ]:
# 获取块结构
result = client.api(
    'GET',
    f'/docx/v1/documents/{ADV_DOC_ID}/blocks/{ADV_DOC_ID}/children',
    params={'page_size': 500}
)

items = result.get('items', [])
print(f"文档共有 {len(items)} 个块\n")

for item in items:
    bt = item['block_type']
    name = block_type_name(bt)
    text = extract_text_from_block(item)
    print(f"  [{name:12}] {text[:50]}")

## 4.3 读取文档元数据

In [ ]:
result = client.api('GET', f'/docx/v1/documents/{ADV_DOC_ID}')
print(json.dumps(result, indent=2, ensure_ascii=False))

## 4.4 错误处理测试

In [ ]:
# 测试 1: 访问不存在的文档
try:
    client.api('GET', '/docx/v1/documents/NOTEXIST/raw_content')
except RuntimeError as e:
    print(f"✓ 预期错误 (不存在的文档): {e}")

In [ ]:
# 测试 2: 无效参数
try:
    client.api('POST', '/docx/v1/documents', json_data={})
except RuntimeError as e:
    print(f"✓ 预期错误 (无效参数): {e}")

In [ ]:
# 测试 3: 更新不存在的块
try:
    client.api(
        'PATCH',
        f'/docx/v1/documents/{ADV_DOC_ID}/blocks/NOTEXIST',
        json_data={'block_type': 2, 'text': {'elements': []}}
    )
except RuntimeError as e:
    print(f"✓ 预期错误 (不存在的块): {e}")